PREPROCESSING FOR FILTERING

In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from datetime import datetime
import traceback
from scipy.signal import butter, sosfilt
import soundfile as sf
import pandas as pd
import os
import matplotlib.dates as mdates
import sys
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import noisereduce as nr
import shutil

c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

NOISE_REDUCTION=False
noise_reduction_str="[NOISE_REDUCTION]" * NOISE_REDUCTION
HIGH_PASS_FILTER=False
high_pass_filter_str="[HIGH_PASS_FILTER]" * HIGH_PASS_FILTER

INPUT_DIR = Path("AudioSamples")
#CSV_INPUT_PATH = Path("audio_samples_metadata.csv")
NOISE_PATH = Path("noise.wav")
OUTPUT_DIR_PREPROCESSED = Path(f"{noise_reduction_str}{high_pass_filter_str}AudioSamplesPreprocessed_ForFiltering")
OUTPUT_DIR_PREPROCESSED.mkdir(parents=True, exist_ok=True)
#OUTPUT_DIR_PREPROCESSED_NORMALIZED = Path("AudioSamplesPreprocessedNormalized")
#OUTPUT_DIR_PREPROCESSED_NORMALIZED.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
FRAME_SIZE = 2048
HOP_LENGTH = 512
TARGET_DB = -1.0
target_amplitude = librosa.db_to_amplitude(TARGET_DB)
CUTOFF_FREQ = 100
NOISE_REDUCTION_PROPORTION = 0.8


In [3]:
if OUTPUT_DIR_PREPROCESSED.exists():
    shutil.rmtree(OUTPUT_DIR_PREPROCESSED)  # Rimuove la cartella e tutto il contenuto

OUTPUT_DIR_PREPROCESSED.mkdir(parents=True) # La ricrea vuota

In [4]:
def highpassfilter(data, cutoff, sr, order=5):
    # Nyquist frequency è la metà della frequenza di campionamento
    nyq = 0.5 * sr
    normal_cutoff = cutoff / nyq
    
    # Crea il filtro in formato SOS (Second-Order Sections) 
    # È più stabile numericamente rispetto al formato classico
    sos = butter(order, normal_cutoff, btype='high', analog=False, output='sos')
    
    # Applica il filtro
    filtered_data = sosfilt(sos, data)
    return filtered_data

In [5]:
def preprocess(audio, offset_to_cut=0.5, high_pass_filter=False, normalize=False, noise_signal=None):
    # 1. RIMOZIONE COMPONENTE DC
    print("Rimozione componente DC...")
    audio = audio - np.mean(audio)

    # 2. FILTRO PASSA ALTO
    if high_pass_filter:
        print("Applicazione filtro passa alto...")
        audio = highpassfilter(audio, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=5)

    # 3. TAGLIO
    print(f"Taglio {offset_to_cut} secondi dall'inizio e dalla fine...")
    offset = int(offset_to_cut * SAMPLE_RATE)
    audio = audio[offset:-offset]

    if noise_signal is not None:
        print("Riduzione del rumore...")
        noise_signal = noise_signal - np.mean(noise_signal)
        if high_pass_filter:
            noise_signal = highpassfilter(noise_signal, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=5)
        noise_signal =  noise_signal[offset:-offset]
        audio = nr.reduce_noise(y=audio, sr=SAMPLE_RATE, y_noise=noise_signal, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, prop_decrease=NOISE_REDUCTION_PROPORTION, stationary=False)
    
    if normalize:
        print("Normalizzazione audio...")
        audio = librosa.util.normalize(audio) * target_amplitude
    return audio

In [6]:
audio_files = list(INPUT_DIR.glob("*.wav"))
total_files = len(audio_files)

In [7]:
#noise, _ = librosa.load(NOISE_PATH, sr=SAMPLE_RATE)

# 1. RIMOZIONE COMPONENTE DC 
#noise = noise - np.mean(noise)

In [8]:
for i, filepath in enumerate(audio_files, 1):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)

    noise = noise if NOISE_REDUCTION else None

    preprocessed_audio=preprocess(audio, offset_to_cut=0.5, high_pass_filter=HIGH_PASS_FILTER, normalize=False, noise_signal=noise)    #normalize=False e noise_signal=noise
    #normalized_preprocessed_audio=preprocess(audio, offset_to_cut=0.5, normalize=True, noise_signal=noise)
    
    output_path = OUTPUT_DIR_PREPROCESSED / filepath.name
    #normalized_output_path = OUTPUT_DIR_PREPROCESSED_NORMALIZED / filepath.name

    print(f"Audio: MIN: {np.min(preprocessed_audio):.4f}, MAX: {np.max(preprocessed_audio):.4f}")

    sf.write(output_path, preprocessed_audio, SAMPLE_RATE, subtype='FLOAT')

    print(f"[{i}/{total_files}] Salvato: {output_path}")
    

Rimozione componente DC...
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.1924, MAX: 0.1346
[1/2766] Salvato: AudioSamplesPreprocessed_ForFiltering\audio_2026-02-23T09-24-36.wav
Rimozione componente DC...
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.4694, MAX: 0.3858
[2/2766] Salvato: AudioSamplesPreprocessed_ForFiltering\audio_2026-02-23T09-25-47.wav
Rimozione componente DC...
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.0329, MAX: 0.0383
[3/2766] Salvato: AudioSamplesPreprocessed_ForFiltering\audio_2026-02-23T09-26-59.wav
Rimozione componente DC...
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.0365, MAX: 0.0366
[4/2766] Salvato: AudioSamplesPreprocessed_ForFiltering\audio_2026-02-23T09-28-11.wav
Rimozione componente DC...
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.1581, MAX: 0.1596
[5/2766] Salvato: AudioSamplesPreprocessed_ForFiltering\audio_2026-02-23T09-29-22.wav
Rimozione componente DC...
Taglio 0